# 🏠 Análisis de Precios de Airbnb
## Modelo de Regresión Lineal Múltiple

---

### Introducción

El mercado de alquileres a corto plazo ha crecido enormemente en los últimos años, y plataformas como **Airbnb** concentran millones de listados con precios muy variados. Entender qué características físicas de una propiedad explican su precio es valioso tanto para anfitriones que quieren fijar tarifas competitivas como para viajeros que buscan opciones razonables.

En este análisis se utiliza un conjunto de datos con **74,111 listados de Airbnb** de distintas ciudades de Estados Unidos. Se construye un **modelo de regresión lineal múltiple** para predecir el precio (en escala logarítmica, `log_price`) a partir de características numéricas del alojamiento.

> **Variable dependiente:** `log_price` — logaritmo natural del precio por noche en USD.  
> **Variables independientes:** capacidad de huéspedes, baños, recámaras, camas, número de reseñas y calificación promedio.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Librerías cargadas correctamente ✅')

Librerías cargadas correctamente ✅


---
## 1. Limpieza de Datos

Antes de construir cualquier modelo, es indispensable asegurar que los datos estén en buen estado. En este dataset encontramos dos problemas principales:

- **Valores faltantes** en columnas como `bathrooms`, `bedrooms`, `beds` y `review_scores_rating`.
- **Valores atípicos extremos** que no representan propiedades reales (por ejemplo, precios de \$0 o propiedades con 18 camas).

**Decisiones tomadas:**
- Los valores faltantes se imputaron con la **mediana** de cada columna, ya que la mediana es robusta ante outliers y preserva la distribución central.
- Se eliminaron registros en el **percentil <1% y >99%** de cada variable para descartar valores claramente incorrectos o casos extremos irreales.

In [2]:
df = pd.read_csv('train.csv')
print(f'Shape original: {df.shape}')

num_cols = ['log_price', 'accommodates', 'bathrooms', 'bedrooms',
            'beds', 'number_of_reviews', 'review_scores_rating']

df_clean = df[num_cols].copy()

# Imputación con mediana
print('\nImputación de valores faltantes:')
for col in df_clean.columns:
    missing = df_clean[col].isna().sum()
    if missing > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'  {col}: {missing} valores → mediana = {median_val}')

# Eliminación de outliers extremos
before = len(df_clean)
for col in df_clean.columns:
    Q1 = df_clean[col].quantile(0.01)
    Q3 = df_clean[col].quantile(0.99)
    df_clean = df_clean[(df_clean[col] >= Q1) & (df_clean[col] <= Q3)]

print(f'\nFilas eliminadas por outliers: {before - len(df_clean)}')
print(f'Shape final: {df_clean.shape}')

Shape original: (74111, 29)

Imputación de valores faltantes:
  bathrooms: 200 valores → mediana = 1.0
  bedrooms: 91 valores → mediana = 1.0
  beds: 131 valores → mediana = 1.0
  review_scores_rating: 16722 valores → mediana = 96.0

Filas eliminadas por outliers: 5175
Shape final: (68936, 7)


---
## 2. Variables del Modelo

| Rol | Variable | Descripción |
|-----|----------|-------------|
| **Dependiente** | `log_price` | Logaritmo del precio por noche |
| Independiente | `accommodates` | Número máximo de huéspedes |
| Independiente | `bathrooms` | Número de baños |
| Independiente | `bedrooms` | Número de recámaras |
| Independiente | `beds` | Número de camas |
| Independiente | `number_of_reviews` | Total de reseñas |
| Independiente | `review_scores_rating` | Calificación promedio (0–100) |

Se usa `log_price` como variable dependiente porque la distribución de precios es asimétrica (sesgo positivo). Aplicar el logaritmo la acerca a una distribución normal, lo cual es un supuesto importante de la regresión lineal.

In [ ]:
target = 'log_price'
features = ['accommodates', 'bathrooms', 'bedrooms',
            'beds', 'number_of_reviews', 'review_scores_rating']

X = df_clean[features]
y = df_clean[target]

# Visualización de la distribución de log_price
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.exp(y), bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución del Precio (USD)', fontweight='bold')
axes[0].set_xlabel('Precio por noche (USD)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(y, bins=60, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribución de log_price (transformado)', fontweight='bold')
axes[1].set_xlabel('log(precio)')
axes[1].set_ylabel('Frecuencia')

plt.suptitle('Efecto de la transformación logarítmica en el precio', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print('La transformación logarítmica normaliza la distribución del precio ✅')

---
## 3. Selección de Características

Para decidir qué variables incluir, se analiza la **correlación de cada característica con `log_price`**. Una correlación alta (positiva o negativa) indica que la variable tiene poder predictivo sobre el precio.

**Hallazgo clave:** Las características relacionadas con el **tamaño y capacidad** del alojamiento (`accommodates`, `beds`, `bedrooms`) muestran correlaciones significativamente más altas que métricas de reputación como `number_of_reviews`.

In [ ]:
corr_with_target = df_clean.corr()[target].drop(target).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target]
bars = ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlación de Pearson con log_price')
ax.set_title('Correlación de cada variable con el precio', fontweight='bold')

for bar, val in zip(bars, corr_with_target.values):
    ax.text(val + 0.005 if val >= 0 else val - 0.005,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(corr_with_target.to_string())

---
## 4. Análisis de Correlación — Pairplot

El **pairplot** permite visualizar simultáneamente las relaciones entre todas las variables del modelo. Cada gráfico de dispersión muestra cómo se relacionan dos variables entre sí, mientras que la diagonal muestra la distribución individual de cada una.

**Observaciones:**
- `accommodates`, `bedrooms` y `beds` muestran una tendencia positiva clara con `log_price`: a mayor capacidad, mayor precio.
- `accommodates` y `beds` están fuertemente correlacionadas entre sí (lo cual revisaremos con el VIF).
- `number_of_reviews` y `review_scores_rating` no muestran una relación lineal clara con el precio.

In [ ]:
sample = df_clean.sample(n=1500, random_state=42)

fig = sns.pairplot(
    sample,
    vars=[target] + features,
    diag_kind='kde',
    plot_kws={'alpha': 0.3, 's': 10, 'color': '#4C72B0'},
    diag_kws={'color': '#4C72B0'}
)
fig.figure.suptitle('Pairplot — Relaciones entre variables (muestra de 1,500 obs.)',
                     y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 5. División Entrenamiento / Prueba

Se divide el dataset en dos subconjuntos:
- **80% entrenamiento** → el modelo aprende los patrones de estos datos.
- **20% prueba** → se evalúa qué tan bien generaliza el modelo a datos que nunca vio.

Esta separación es fundamental para detectar **sobreajuste** (*overfitting*): si el modelo tiene un desempeño mucho mejor en entrenamiento que en prueba, significa que memorizó los datos en lugar de aprender patrones generales.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Entrenamiento : {X_train.shape[0]:,} filas ({100*len(X_train)/len(X):.0f}%)')
print(f'Prueba        : {X_test.shape[0]:,} filas ({100*len(X_test)/len(X):.0f}%)')

---
## 6. Construcción del Modelo — Multicolinealidad (VIF)

Antes de entrenar, se verifica la **multicolinealidad**: si dos variables independientes están muy correlacionadas entre sí, el modelo no puede distinguir el efecto individual de cada una, lo que inestabiliza los coeficientes.

Se calcula el **Factor de Inflación de Varianza (VIF)** para cada variable:
- **VIF < 5** → sin problema de multicolinealidad.
- **VIF entre 5 y 10** → multicolinealidad moderada.
- **VIF > 10** → multicolinealidad severa, considerar eliminar la variable.

El VIF se calcula regresando cada variable independiente contra todas las demás y evaluando el R² resultante: `VIF = 1 / (1 - R²)`.

In [ ]:
# Cálculo de VIF con sklearn (sin statsmodels)
vif_rows = []
for col in features:
    X_otros = X_train.drop(columns=col)
    r2 = LinearRegression().fit(X_otros, X_train[col]).score(X_otros, X_train[col])
    vif = 1 / (1 - r2) if r2 < 1 else float('inf')
    vif_rows.append({'Variable': col, 'VIF': round(vif, 4)})

vif_data = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors_vif = ['#e74c3c' if v >= 5 else '#2ecc71' for v in vif_data['VIF']]
ax.barh(vif_data['Variable'], vif_data['VIF'], color=colors_vif, edgecolor='white')
ax.axvline(5, color='red', linestyle='--', linewidth=1.2, label='Umbral VIF = 5')
ax.set_xlabel('VIF')
ax.set_title('Factor de Inflación de Varianza (VIF) por Variable', fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(vif_data.to_string(index=False))
print('\n✅ Todos los VIF < 5: no hay multicolinealidad problemática.')

In [ ]:
# Entrenamiento del modelo
model = LinearRegression()
model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'Variable': features,
    'Coeficiente': model.coef_
}).sort_values('Coeficiente', ascending=False)

print(f'Intercepto: {model.intercept_:.4f}')
print('\nCoeficientes del modelo:')
print(coef_df.to_string(index=False))

# Gráfico de coeficientes
fig, ax = plt.subplots(figsize=(8, 4))
colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coeficiente']]
ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente')
ax.set_title('Coeficientes del Modelo de Regresión Lineal Múltiple', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación de los Coeficientes

Dado que la variable dependiente está en escala **logarítmica**, los coeficientes se interpretan como **cambios porcentuales aproximados** en el precio:

| Variable | Coeficiente | Interpretación |
|----------|-------------|----------------|
| `accommodates` | +0.193 | Cada huésped adicional que puede alojar la propiedad **incrementa el precio ~19.3%** |
| `bathrooms` | +0.073 | Cada baño adicional **incrementa el precio ~7.3%** |
| `bedrooms` | +0.042 | Cada recámara adicional **incrementa el precio ~4.2%** |
| `review_scores_rating` | +0.009 | Una calificación más alta tiene efecto positivo pero **muy pequeño** |
| `beds` | −0.009 | Efecto levemente negativo una vez controlando por las otras variables |
| `number_of_reviews` | −0.001 | Prácticamente **sin efecto** sobre el precio |

> **Nota sobre `beds`:** Su coeficiente negativo no significa que más camas baje el precio. Se debe a que `beds` y `accommodates` están correlacionadas — una vez que el modelo ya conoce la capacidad de huéspedes, agregar más camas sin más cuartos o baños no aumenta el precio.

---
## 7. Evaluación del Modelo

Se evalúa el modelo con dos métricas principales:

- **R² (coeficiente de determinación):** Proporción de la varianza del precio que explica el modelo. Un R² = 1.0 sería predicción perfecta; R² = 0 sería igual de malo que predecir siempre el promedio.
- **MSE / RMSE:** Error cuadrático medio y su raíz cuadrada. Miden la magnitud promedio del error de predicción.

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test  = mean_squared_error(y_test,  y_pred_test)
rmse_test = np.sqrt(mse_test)

# Tabla de métricas
metrics = pd.DataFrame({
    'Métrica': ['R²', 'MSE', 'RMSE'],
    'Entrenamiento': [f'{r2_train:.4f}', f'{mse_train:.4f}', f'{np.sqrt(mse_train):.4f}'],
    'Prueba':        [f'{r2_test:.4f}',  f'{mse_test:.4f}',  f'{rmse_test:.4f}']
})
print(metrics.to_string(index=False))

### Interpretación de las Métricas

**R² = 0.35** — El modelo explica el **35% de la varianza** en el precio de los listados de Airbnb usando solo 6 variables numéricas. 

¿Es bueno este resultado? En el contexto de precios de alojamiento, es un punto de partida **razonable pero mejorable**. Las razones por las que el R² no es más alto son:

1. **Variables categóricas excluidas:** La ciudad, el tipo de propiedad, el tipo de habitación y el vecindario tienen un impacto enorme en el precio y no se incluyeron en este modelo básico.
2. **Factores no observados:** Amenidades, calidad de las fotos, políticas de cancelación y factores estacionales afectan el precio pero son difíciles de cuantificar.
3. **Linealidad:** Un modelo lineal puede no capturar relaciones no lineales entre las variables y el precio.

**Sin sobreajuste:** La diferencia entre R² entrenamiento (0.347) y prueba (0.351) es mínima, lo que confirma que el modelo **generaliza bien** a datos nuevos.

**RMSE = 0.493** en escala logarítmica indica que el error promedio de predicción es de aproximadamente ±50% del precio real, lo cual es consistente con un R² de 0.35.

---
## 8. Predicciones vs Valores Reales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evaluación del Modelo — Predicciones vs Valores Reales',
             fontsize=14, fontweight='bold')

# Scatter predicciones vs reales
ax = axes[0]
ax.scatter(y_test, y_pred_test, alpha=0.2, s=10, color='#4C72B0')
min_v, max_v = y_test.min(), y_test.max()
ax.plot([min_v, max_v], [min_v, max_v], 'r--', lw=1.5, label='Predicción perfecta')
ax.set_xlabel('Valores Reales (log_price)')
ax.set_ylabel('Predicciones (log_price)')
ax.set_title(f'Predicciones vs Reales  (R² = {r2_test:.3f})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Distribución de residuales
residuals = y_test.values - y_pred_test
ax2 = axes[1]
ax2.hist(residuals, bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
ax2.axvline(0, color='red', linestyle='--', lw=1.5, label='Error = 0')
ax2.set_xlabel('Residual (Real − Predicción)')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Distribución de Residuales')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretación de las gráficas

**Gráfica izquierda — Predicciones vs Reales:**  
Los puntos idealmente deberían alinearse sobre la línea roja discontinua (predicción perfecta). Se observa que el modelo captura bien la tendencia central, pero hay dispersión considerable, especialmente en precios altos y bajos. Esto es consistente con un R² de 0.35.

**Gráfica derecha — Distribución de Residuales:**  
Los residuales (diferencia entre valor real y predicción) se distribuyen de forma **aproximadamente normal y centrada en cero**, lo cual es un buen signo: indica que el modelo no tiene sesgo sistemático. No sobreestima ni subestima consistentemente.

---
## 9. Error Cuadrático Medio (MSE)

In [ ]:
print('=' * 50)
print('RESUMEN DE ERRORES DEL MODELO')
print('=' * 50)
print(f'MSE  (prueba) = {mse_test:.6f}')
print(f'RMSE (prueba) = {rmse_test:.6f}')
print()

# Muestra de predicciones vs reales
resultados = pd.DataFrame({
    'Real (log)':       y_test.values[:10].round(4),
    'Predicción (log)': y_pred_test[:10].round(4),
    'Real (USD)':       np.exp(y_test.values[:10]).round(2),
    'Predicción (USD)': np.exp(y_pred_test[:10]).round(2),
    'Error (USD)':      (np.exp(y_test.values[:10]) - np.exp(y_pred_test[:10])).round(2)
})
print('Primeras 10 predicciones:')
print(resultados.to_string(index=False))

---
## Conclusiones

### ¿Qué aprendimos sobre los precios de Airbnb?

1. **La capacidad de la propiedad es el principal driver de precio.** La variable `accommodates` tiene la correlación más alta con el precio (0.58) y el coeficiente más grande en el modelo (+0.193). Esto tiene sentido intuitivo: una propiedad que aloja más personas puede cobrar más porque distribuye el costo entre más huéspedes.

2. **El número de baños y recámaras también importa**, aunque en menor medida. Cada baño adicional incrementa el precio ~7.3% y cada recámara ~4.2%, reflejando el valor que los huéspedes dan a la privacidad y comodidad.

3. **Las reseñas tienen poco efecto sobre el precio.** Tanto el volumen de reseñas como la calificación promedio tienen correlaciones muy bajas con el precio. Esto sugiere que los anfitriones no ajustan sus precios significativamente en función de sus calificaciones.

4. **El modelo explica el 35% de la varianza**, lo cual es un resultado razonable para un modelo lineal con solo variables numéricas. Para mejorar significativamente se necesitaría incorporar variables como ciudad, tipo de propiedad, amenidades y factores estacionales.

5. **No hay sobreajuste**: el rendimiento en entrenamiento y prueba es prácticamente idéntico, lo que confirma que el modelo generaliza bien.

### ¿Qué se podría mejorar?

- Codificar variables categóricas como `city`, `property_type`, `room_type` y `cancellation_policy`.
- Probar modelos no lineales como Random Forest o Gradient Boosting que capturan interacciones entre variables.
- Incorporar la cantidad de amenidades (el campo `amenities` contiene listas de servicios).
- Aplicar ingeniería de características, como calcular la antigüedad del anfitrión o la época del año.